In [1]:
import pandas as pd

from google.cloud import bigquery

# Initialize client (replace with your GCP project ID)
project_id = "lewagonyining"
client = bigquery.Client(project=project_id)

# Define your SQL query
query = """
    SELECT *
    FROM `lewagonyining.tiller.order_data`
"""

# Run query and load into a Pandas DataFrame
df_order_data = client.query(query).to_dataframe()

# Display results
df_order_data.head()

/home/yiningl/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id_order,id_store,id_table,id_waiter,id_device,date_opened,date_closed,dim_status,dim_source,m_nb_customer,m_cached_payed,m_cached_price
0,112572333,7786,<NA>,24920,14854,2020-01-12 16:03:02+00:00,2020-01-12 16:03:13+00:00,CLOSED,tiller-order,1,11.9,11.9
1,113129706,8291,<NA>,16823,15760,2020-01-16 11:11:02+00:00,2020-01-16 11:11:30+00:00,CLOSED,tiller-order,1,11.7,11.7
2,50451573,6008,<NA>,10990,11065,2018-11-27 18:57:58+00:00,2018-11-27 18:58:59+00:00,CLOSED,Tiller iPAD,1,25.0,25.0
3,50451883,730,8292,1095,754,2018-11-27 19:01:28+00:00,2018-11-27 19:06:29+00:00,CLOSED,Tiller iPAD,1,28.8,28.8
4,50454421,351,19304,461,128,2018-11-27 19:17:04+00:00,2018-11-27 21:18:41+00:00,CLOSED,Tiller iPAD,2,55.0,55.0


In [2]:
# Export DataFrame to CSV inside the data folder
df_order_data.to_csv("data/df_order_data.csv", index=False)

In [ ]:
# Import other tables
query = """
    SELECT
        id_order_line,
        id_order,
        date_opended AS date_opened,
        m_quantity,
        m_unit_price,
        m_total_price_inc_vat,
        m_tax_percent,
        m_discount_amount,
        dim_type,
        dim_status,
        dim_category_translated,
        dim_name_translated
    FROM `lewagonyining.tiller.order_line`
    ORDER BY date_opened ASC
"""

# Run query and load into a Pandas DataFrame
df_order_line = client.query(query).to_dataframe()

df_order_line.to_csv("data/df_order_line.csv", index=False)

In [4]:
# Import other tables
query = """
    SELECT *
    FROM `lewagonyining.tiller.payment_data`
"""

# Run query and load into a Pandas DataFrame
df_payment_data = client.query(query).to_dataframe()

df_payment_data.to_csv("data/df_payment_data.csv", index=False)

In [5]:
# Import other tables
query = """
    SELECT *
    FROM `lewagonyining.tiller.store_data`
"""

# Run query and load into a Pandas DataFrame
df_store_data = client.query(query).to_dataframe()

df_store_data.to_csv("data/df_store_data.csv", index=False)

/home/yiningl/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [14]:
print("df_order_line columns:", df_order_line.columns.tolist())
print("df_order_data columns:", df_order_data.columns.tolist())

df_order_line columns: ['id_order_line', 'id_order', 'date_opened', 'm_quantity', 'm_unit_price', 'm_total_price_inc_vat', 'm_tax_percent', 'm_discount_amount', 'dim_type', 'dim_status', 'dim_category_translated', 'dim_name_translated']
df_order_data columns: ['id_order', 'id_store', 'id_table', 'id_waiter', 'id_device', 'date_opened', 'date_closed', 'dim_status', 'dim_source', 'm_nb_customer', 'm_cached_payed', 'm_cached_price']


In [2]:
import pandas as pd

# -------------------------------------------------------------------------
# 1. Load Data & Standardise Column Names
# -------------------------------------------------------------------------
df_order_data = pd.read_csv("data/df_order_data.csv")
df_order_line = pd.read_csv("data/df_order_line.csv")

# Clean column headers (removes whitespace and forces lowercase)
df_order_data.columns = df_order_data.columns.str.strip().str.lower()
df_order_line.columns = df_order_line.columns.str.strip().str.lower()


# -------------------------------------------------------------------------
# 2. Add Macro Category Mapping
# -------------------------------------------------------------------------
def classify_macro_category(category_name):
    """Maps granular restaurant categories into macro groups."""
    cat = str(category_name).lower().strip()

    beverage_keywords = [
        "drink",
        "boisson",
        "wine",
        "vin",
        "beer",
        "biere",
        "bière",
        "pressure",
        "cocktail",
        "water",
        "eau",
        "tea",
        "thé",
        "coffee",
        "café",
        "lassi",
        "digestif",
        "champagne",
        "soft",
        "san pellegrino",
        "evian",
        "whisky",
        "martini",
    ]

    dessert_keywords = [
        "dessert",
        "sorbet",
        "ice cream",
        "patisserie",
        "pastries",
        "sugar",
        "sweet",
    ]

    starter_keywords = [
        "starter",
        "entree",
        "entrée",
        "appetizer",
        "soup",
        "salad",
    ]

    if any(k in cat for k in beverage_keywords):
        return "Beverages"
    elif any(k in cat for k in dessert_keywords):
        return "Desserts"
    elif any(k in cat for k in starter_keywords):
        return "Starters"
    else:
        return "Mains / Other"


# Create the macro_category column on df_order_line
df_order_line["macro_category"] = df_order_line[
    "dim_category_translated"
].apply(classify_macro_category)


# -------------------------------------------------------------------------
# 3. KPI Calculation Functions
# -------------------------------------------------------------------------
def calculate_kpis(df_order_data: pd.DataFrame, df_order_line: pd.DataFrame):
    """Calculates high-level revenue and operational KPIs."""
    merged = pd.merge(df_order_line, df_order_data, on="id_order", how="inner")

    # --- KPI 1: Total Revenue (Including VAT) ---
    total_revenue_inc_vat = merged["m_total_price_inc_vat"].sum()

    # --- KPI 2: Average Order Value (AOV) ---
    total_orders = df_order_data["id_order"].nunique()
    aov = total_revenue_inc_vat / total_orders if total_orders > 0 else 0

    # --- KPI 3: Average Revenue Per Cover (Spend per Head) ---
    total_customers = df_order_data["m_nb_customer"].sum()
    rev_per_cover = (
        total_revenue_inc_vat / total_customers if total_customers > 0 else 0
    )

    # --- KPI 4: Discount Impact Ratio ---
    total_discounts = df_order_line["m_discount_amount"].sum()
    gross_revenue = total_revenue_inc_vat + total_discounts
    discount_impact_ratio = (
        (total_discounts / gross_revenue) if gross_revenue > 0 else 0
    )

    return {
        "Total Revenue (Inc VAT)": round(total_revenue_inc_vat, 2),
        "AOV": round(aov, 2),
        "Spend Per Head": round(rev_per_cover, 2),
        "Discount Impact Ratio (%)": round(discount_impact_ratio * 100, 2),
    }


def get_top_bottom_performers(df_order_line: pd.DataFrame, top_n: int = 5):
    """Calculates top and bottom performing product items while filtering out deposits and adjustments."""
    df_products = df_order_line.copy()

    # Filter for products only
    if "dim_type" in df_products.columns:
        df_products = df_products[
            df_products["dim_type"].astype(str).str.strip().str.lower()
            == "product"
        ]

    # Exclude deposit items and order adjustments
    exclude_terms = ["DEPOSIT", "CONSIGNE", "ORDER", "DELIVERY PACKAGE"]
    pattern = "|".join(exclude_terms)

    df_products = df_products[
        ~df_products["dim_name_translated"]
        .astype(str)
        .str.upper()
        .str.contains(pattern)
        & ~df_products["dim_category_translated"]
        .astype(str)
        .str.upper()
        .str.contains(pattern)
    ]

    performers = (
        df_products.groupby(
            ["macro_category", "dim_name_translated"]
        )
        .agg(
            total_quantity_sold=("m_quantity", "sum"),
            total_revenue=("m_total_price_inc_vat", "sum"),
        )
        .reset_index()
    )

    top_performers = performers.sort_values(
        by="total_quantity_sold", ascending=False
    ).head(top_n)

    bottom_performers = performers.sort_values(
        by="total_quantity_sold", ascending=True
    ).head(top_n)

    return top_performers, bottom_performers


def calculate_attachment_rates(
    df_order_line: pd.DataFrame, target_macro: str = "Beverages"
):
    """Calculates what percentage of total orders contain at least one item from the target macro category."""
    order_macros = (
        df_order_line.groupby("id_order")["macro_category"]
        .apply(set)
        .reset_index(name="macros")
    )

    total_orders = len(order_macros)
    if total_orders == 0:
        return 0.0

    orders_with_target = order_macros["macros"].apply(
        lambda m_set: target_macro in m_set
    )

    attachment_rate = (orders_with_target.sum() / total_orders) * 100
    return round(attachment_rate, 2)


# -------------------------------------------------------------------------
# 4. Execution
# -------------------------------------------------------------------------
kpi_results = calculate_kpis(
    df_order_data=df_order_data, df_order_line=df_order_line
)

top_items, bottom_items = get_top_bottom_performers(
    df_order_line, top_n=5
)

beverage_attach_rate = calculate_attachment_rates(
    df_order_line, target_macro="Beverages"
)

print("KPI Overview:", kpi_results)
print("\nTop 5 Performers:\n", top_items)
print("\nBottom 5 Performers:\n", bottom_items)
print("\nBeverage Attachment Rate:", f"{beverage_attach_rate}%")

KPI Overview: {'Total Revenue (Inc VAT)': 20305231.78, 'AOV': 15.85, 'Spend Per Head': 13.25, 'Discount Impact Ratio (%)': 0.55}

Top 5 Performers:
    macro_category            dim_name_translated  total_quantity_sold  \
40      Beverages            50 CL BLONDE METEOR             237073.0   
39      Beverages           50 CL BLANCHE METEOR              62926.0   
41      Beverages  50 CL BLONDE TIGER BOCK 50 CL              58026.0   
28      Beverages            25 CL BLONDE METEOR              41171.0   
7       Beverages            1.5 L BLONDE METEOR              33646.0   

    total_revenue  
40     1417966.90  
39      439168.85  
41      348111.00  
28      143403.40  
7       605278.80  

Bottom 5 Performers:
      macro_category          dim_name_translated  total_quantity_sold  \
3074  Mains / Other  Bigorre black sausage slice                0.036   
8984  Mains / Other                      mothais                0.040   
8955  Mains / Other                    monk head  

In [4]:
print(df_order_line["dim_category_translated"].dropna().unique())

['Evening' 'Sugar' 'VARIOUS' 'Drinks Rest' 'come' 'Corkage' 'Meal Jars'
 'Earth' 'Vegetables' 'Soft groceries' 'Midi' 'Condiments' 'Beer Grocery'
 'Brunch' 'Indian wines' 'Pains' 'Southern dishes' 'Cold drinks'
 'Place North' 'Beers' 'Cocktail' 'Indian drinks' 'Hot drinks' 'Starters'
 'Menu 16,50' 'Desserts' 'Appetizers' 'Place Vegetarian' 'White wines'
 'Menu 25,00' 'North Vegetarian Choice' 'Red Wines' 'pitcher wine'
 'Southern Vegetarian Choice' 'Evening menu' 'Rice' 'Rosé Wines' 'More'
 'digestive' 'New Year&#39;s Eve menu' 'Menu midi' 'Valentine&#39;s Day'
 'Appetizers / Soups / Salads' 'Dishes / Chicken'
 'Tibetan specialties Starter' 'Bordeaux Red PDO' 'Royal biryani' 'Pain'
 'Fish - Shrimps - Prawns' 'Drinks' 'Sagarmatha starter mix'
 'Tandoor specialty starters' 'Dishes / Lamb' 'Tibetan Specialties Dishes'
 'PDO Red Wines' 'Biriyani' 'Bourgueil Le Plessis'
 'Wine by the glass 14 cl' 'Rose wines AOP' 'tavel' 'Digestives 2 cl'
 'Traditional Nepalese dish' 'Waters' 'Lassi' 'San P

In [3]:
def classify_macro_category(category_name):
    cat = str(category_name).lower().strip()

    # Keywords for Beverages
    beverage_keywords = [
        "drink", "boisson", "wine", "vin", "beer", "biere", "bière", "pressure",
        "cocktail", "water", "eau", "tea", "thé", "coffee", "café", "lassi",
        "digestif", "champagne", "soft", "san pellegrino", "evian", "whisky", "martini"
    ]

    # Keywords for Desserts
    dessert_keywords = ["dessert", "sorbet", "ice cream", "patisserie", "pastries", "sugar", "sweet"]

    # Keywords for Starters
    starter_keywords = ["starter", "entree", "entrée", "appetizer", "soup", "salad"]

    if any(k in cat for k in beverage_keywords):
        return "Beverages"
    elif any(k in cat for k in dessert_keywords):
        return "Desserts"
    elif any(k in cat for k in starter_keywords):
        return "Starters"
    else:
        return "Mains / Other"

# Apply mapping to a new column
df_order_line["macro_category"] = df_order_line["dim_category_translated"].apply(classify_macro_category)

In [5]:
import pandas as pd

# -------------------------------------------------------------------------
# 1. Load Data & Standardise Column Names
# -------------------------------------------------------------------------
df_order_data = pd.read_csv("data/df_order_data.csv")
df_order_line = pd.read_csv("data/df_order_line.csv")

# Clean column headers
df_order_data.columns = df_order_data.columns.str.strip().str.lower()
df_order_line.columns = df_order_line.columns.str.strip().str.lower()


# -------------------------------------------------------------------------
# 2. Operational Efficiency KPIs Function
# -------------------------------------------------------------------------
def calculate_operational_kpis(df_order_data: pd.DataFrame):
    """Calculates table dwell time, peak hourly revenue, and table turnover metrics."""
    df_ops = df_order_data.copy()

    # Convert timestamp columns to datetime
    df_ops["date_opened"] = pd.to_datetime(df_ops["date_opened"])
    df_ops["date_closed"] = pd.to_datetime(df_ops["date_closed"])

    # --- KPI 1: Table Dwell Time (Minutes) ---
    df_ops["dwell_time_minutes"] = (
        df_ops["date_closed"] - df_ops["date_opened"]
    ).dt.total_seconds() / 60.0

    # Filter out invalid outliers (e.g. negative duration or orders open over 4 hours)
    valid_orders = df_ops[
        (df_ops["dwell_time_minutes"] > 0) & (df_ops["dwell_time_minutes"] <= 240)
    ]

    avg_dwell_time = valid_orders["dwell_time_minutes"].mean()
    median_dwell_time = valid_orders["dwell_time_minutes"].median()

    # --- KPI 2: Hourly Rush Analysis (Orders & Revenue by Hour) ---
    df_ops["opening_hour"] = df_ops["date_opened"].dt.hour
    df_ops["day_of_week"] = df_ops["date_opened"].dt.day_name()

    # Pick the right revenue column in df_order_data
    revenue_col = (
        "m_cached_payed"
        if "m_cached_payed" in df_ops.columns
        else "m_cached_price"
    )

    hourly_performance = (
        df_ops.groupby("opening_hour")
        .agg(
            total_orders=("id_order", "nunique"),
            total_customers=("m_nb_customer", "sum"),
            total_revenue=(revenue_col, "sum"),
        )
        .reset_index()
        .sort_values(by="total_orders", ascending=False)
    )

    # --- KPI 3: Table Turnover Rate ---
    if "dim_table" in df_ops.columns:
        tables_df = df_ops[
            df_ops["dim_table"].notna()
            & (df_ops["dim_table"].astype(str).str.strip() != "")
        ]
        total_unique_tables = tables_df["dim_table"].nunique()
        total_table_orders = len(tables_df)

        avg_turnover_per_table = (
            total_table_orders / total_unique_tables
            if total_unique_tables > 0
            else 0
        )
    else:
        avg_turnover_per_table = 0.0

    return {
        "Average Dwell Time (Mins)": round(avg_dwell_time, 2),
        "Median Dwell Time (Mins)": round(median_dwell_time, 2),
        "Average Table Turnover Rate": round(avg_turnover_per_table, 2),
        "Hourly Breakdown": hourly_performance,
    }


# -------------------------------------------------------------------------
# 3. Execution
# -------------------------------------------------------------------------
ops_results = calculate_operational_kpis(df_order_data)

print("--- Operational Efficiency Overview ---")
print(f"Average Dwell Time: {ops_results['Average Dwell Time (Mins)']} minutes")
print(f"Median Dwell Time: {ops_results['Median Dwell Time (Mins)']} minutes")
print(f"Average Turnover per Table: {ops_results['Average Table Turnover Rate']} turns")

print("\n--- Top 5 Peak Hours by Order Volume ---")
print(ops_results["Hourly Breakdown"].head(5).to_string(index=False))

--- Operational Efficiency Overview ---
Average Dwell Time: 11.57 minutes
Median Dwell Time: 0.55 minutes
Average Turnover per Table: 0.0 turns

--- Top 5 Peak Hours by Order Volume ---
 opening_hour  total_orders  total_customers  total_revenue
           18        140285           169055     2903277.12
           19        137003           168207     2886081.56
           17        117768           130792     1845624.77
           20        117761           137599     2112287.82
           11        112613           150534     2273115.94
